### Installation

In [ ]:
from google.colab import drive

# 1. Mount Google Drive
# This will trigger a popup asking for permission to access your Drive
drive.mount('/content/drive')

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit", #Specifies which model to download from Hugging Face.
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory (how much text the model can read at once)
    load_in_4bit = True,     # Loads the model in 4-bit quantization. This drastically reduces VRAM usage, allowing you to train on consumer GPUs (like a T4 or RTX 30/40 series).
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We are doing LoRA (Parameter Efficient Fine-Tuning), not retraining the entire brain.
    # token = "..."   # only needed for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128, The "Rank" of the LoRA matrices.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
        use_gradient_checkpointing = "unsloth", # A memory-saving technique specifically optimized by Unsloth to handle longer contexts.
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  
)

<a name="Data"></a>
### Data preparation

Load the BIRD fine-tuning data (`bird_finetune.jsonl`) and format it for supervised fine-tuning (SFT).

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="path/to/bird_finetune.jsonl", split="train")

Let's see the structure of the dataset:

In [ ]:
dataset

We now convert the reasoning dataset into conversational format:

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)
# FORMAT DATASET
def formatting_prompts_func(examples):
    texts = []
    # Zip your specific columns "user" and "sql"
    for user_msg, assistant_msg in zip(examples["user"], examples["sql"]):
        conversation = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
        # Apply the Qwen template to turn the list into a training string
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

# Apply the formatting to create a "text" column
processed_dataset = dataset.map(formatting_prompts_func, batched=True)

print("Success! Data prepared.")

Let's see the first transformed row:

In [ ]:
processed_dataset[0]

Now let's see how long the dataset is:

In [ ]:
print(len(processed_dataset))

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_dataset,
    max_seq_length = 4096,
    dataset_num_proc = 8, 
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4, 
        gradient_accumulation_steps = 4,
        bf16 = True,
        warmup_steps = 30, 
        num_train_epochs = 1, # Set this for 1 full training run.
        #max_steps = 100,
        learning_rate = 2e-5, 
        logging_steps = 1,
        optim = "adamw_8bit", #Uses an 8-bit optimizer to save even more memory.
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        max_grad_norm = 0.3,
        output_dir = "qwen3_sft_output",
        save_strategy = "steps",
        save_steps = 100,
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# This function automatically configures the DataCollator for Qwen's specific template
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# Verify it worked
print("✅ Masking enabled! The model will now ONLY learn from the SQL output.")

In [ ]:
import torch
import gc
def test_memory_usage(batch_size_to_test):
    print(f"\n--- Testing Batch Size: {batch_size_to_test} ---")

    # 1. Clear previous memory
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # 2. Update Trainer Config dynamically
    trainer.args.per_device_train_batch_size = batch_size_to_test
    trainer.args.max_steps = 5  # Run only 5 steps to test
    trainer.args.logging_steps = 1

    # 3. Run a short training loop
    print("Running 5 steps...")
    try:
        trainer.train()
    except Exception as e:
        print(f"❌ OOM (Out of Memory) or Error at BS={batch_size_to_test}: {e}")
        return

    # 4. Check Peak Memory
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print(f"✅ Success! Peak Memory Used: {peak_memory:.2f} GB / {total_memory:.2f} GB")
    print(f"   Usage: {peak_memory/total_memory*100:.1f}%")

# --- EXECUTE TESTS ---
# Test your current setting
test_memory_usage(4)

# Test the recommended setting
test_memory_usage(32)

# Test an aggressive setting (If 32 is easy, this might fit)
test_memory_usage(64)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for reasoning inference are `temperature = 0.6, top_p = 0.95, top_k = 20`

For normal chat based inference, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [ ]:
# 1. Define the Helper Function (Fixed to return the value)
def generate_sql_logic(schema, question):
    # Matches the 'You are a SQL expert...' training format
    return f"""You are a SQL expert. Generate a SQL query to answer the question based on the database schema provided.

{schema}

Hint: Use the column names exactly as they appear in the schema.

Question: {question}"""

# =========================================================
# Example: inference on a custom schema
# =========================================================

# 1. Define your Schema
real_excel_schema = """
CREATE TABLE graduates (
    Name TEXT,
    High_School TEXT,
    University_Attending TEXT,
    State_Country TEXT
);
"""

# 2. Ask a question
user_question = "Which universities have a student count that is higher than the average number of students per university?"

# 3. Generate the Prompt
full_prompt = generate_sql_logic(real_excel_schema, user_question)

# 4. Tokenize
messages = [{"role": "user", "content": full_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# 5. Generate
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    temperature = 0.1,
    top_p = 0.9,
    streamer = text_streamer
)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

Save to Google Drive

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
# This will trigger a popup asking for permission to access your Drive
drive.mount('/content/drive')

# 2. Define your Google Drive path
# You can change 'Unsloth_Qwen_SQL' to whatever folder name you prefer
drive_save_path = "/path/to/qwen_sql_lora"

# Create the directory if it doesn't exist
os.makedirs(drive_save_path, exist_ok=True)

print(f"Saving model to {drive_save_path}...")

# 3. Save the LoRA adapters and tokenizer to Drive
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print("SUCCESS: Model saved to Google Drive! It will be there next time you open Colab.")

Load Model from drive

In [ ]:
from unsloth import FastLanguageModel
from google.colab import drive

# 1. Mount Drive again
drive.mount('/content/drive')

# 2. Point to the folder where you saved it
drive_save_path = "/path/to/qwen_sql_lora"

# 3. Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = drive_save_path, # Load from Drive instead of HuggingFace
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Enable native inference again
FastLanguageModel.for_inference(model)